# A(image_gt) 与 projs 差异可视化
按顺序运行每个小节，生成投影对比与差异热力图。

In [ ]:
import os
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

from src.dataset.tigre import TIGREDataset
from src.loss.vesde_loss import VESDEGuidance

LOG_DIR = Path('./logs')
PATIENT_ID = None  # set to a specific id string if needed
CONFIG_PATH = 'src/sde_configs/ve/IXI_256_ncsnpp_continuous.py'
CKPT_PATH = None  # set to diffusion checkpoint path
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

assert LOG_DIR.exists(), f'LOG_DIR not found: {LOG_DIR}'

In [ ]:
def find_eval_dirs(root: Path):
    return list(root.rglob('eval'))

def pick_eval_dir(eval_dirs, patient_id=None):
    if patient_id is None:
        for d in eval_dirs:
            if (d / 'image_gt.npy').exists():
                return d
        return None
    for d in eval_dirs:
        if patient_id in str(d) and (d / 'image_gt.npy').exists():
            return d
    return None

eval_dirs = find_eval_dirs(LOG_DIR)
EVAL_DIR = pick_eval_dir(eval_dirs, PATIENT_ID)
print('Found eval dirs:', len(eval_dirs))
print('Using eval dir:', EVAL_DIR)
assert EVAL_DIR is not None, 'No eval dir with image_gt.npy found.'

In [ ]:
image_gt = np.load(EVAL_DIR / 'image_gt.npy')
print('image_gt shape:', image_gt.shape)

# locate dataset pickle from parent dirs if available
data_root = EVAL_DIR.parents[1] if len(EVAL_DIR.parents) > 1 else None
print('data_root:', data_root)

# You may need to manually set the pickle path if not in logs tree
PICKLE_PATH = None

In [ ]:
if PICKLE_PATH is None:
    print('Set PICKLE_PATH to your data pickle for rays/projs.')
else:
    dset = TIGREDataset(PICKLE_PATH, n_rays=4096, type='val', device=DEVICE)
    rays = dset.rays[0].reshape(-1, 8).to(DEVICE)
    gt_projs = dset.projs[0].reshape(-1).to(DEVICE)
    bound_box = torch.tensor([dset.sVoxel[0]/2, dset.sVoxel[1]/2, dset.sVoxel[2]/2], device=DEVICE)

    assert CKPT_PATH is not None, 'Set CKPT_PATH to diffusion checkpoint.'
    vesde = VESDEGuidance(CONFIG_PATH, CKPT_PATH, device=DEVICE)

    vol = torch.tensor(image_gt, dtype=torch.float32, device=DEVICE)[None, None]
    proj_pred = vesde._project_volume_with_rays(vol, rays, bound_box, n_samples=512, rays_chunk=4096)

    mse = torch.mean((proj_pred - gt_projs) ** 2).item()
    psnr = -10 * np.log10(mse + 1e-12)
    print('MSE:', mse, 'PSNR:', psnr)

    # visualize a small subset
    pred_np = proj_pred.detach().cpu().numpy()
    gt_np = gt_projs.detach().cpu().numpy()
    diff = pred_np - gt_np

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].plot(gt_np[:2000]); axes[0].set_title('GT projs')
    axes[1].plot(pred_np[:2000]); axes[1].set_title('A(image_gt)')
    axes[2].plot(diff[:2000]); axes[2].set_title('Diff')
    plt.tight_layout()
    out_path = EVAL_DIR / 'proj_diff.png'
    fig.savefig(out_path)
    plt.close(fig)
    print('Saved:', out_path)

## 1. 加载依赖与路径配置

In [5]:
import os
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

# 基础路径
BASE = Path("/home/jym/naf_cbct")
LOG_DIR = BASE / "logs" / "20260403_CT_1views_centerPad_noLrSchedule_fixVESDEsag_annealing_sds0.005_startFrom0_interval10_nopaas_fidelity2" / "1.3.6.1.4.1.9328.50.4.0767"
EVAL_DIR = LOG_DIR / "eval"
PICKLE_PATH = Path("/home/public/CTSpine1K/data/data-MHD_ctpro_woMask1/1.3.6.1.4.1.9328.50.4.0767/data_2views.pickle")

# 选择角度索引
EVAL_INDEX = 0

print("Eval dir:", EVAL_DIR)
print("Pickle:", PICKLE_PATH)

Eval dir: /home/jym/naf_cbct/logs/20260403_CT_1views_centerPad_noLrSchedule_fixVESDEsag_annealing_sds0.005_startFrom0_interval10_nopaas_fidelity2/1.3.6.1.4.1.9328.50.4.0767/eval
Pickle: /home/public/CTSpine1K/data/data-MHD_ctpro_woMask1/1.3.6.1.4.1.9328.50.4.0767/data_2views.pickle


In [6]:
# 自动定位 eval 目录（以 image_gt.npy 为准）
if not EVAL_DIR.exists() or not (EVAL_DIR / "image_gt.npy").exists():
    candidates = list((BASE / "logs").rglob("image_gt.npy"))
    print("image_gt.npy candidates:")
    for c in candidates[:20]:
        print(" -", c)
    if len(candidates) == 1:
        EVAL_DIR = candidates[0].parent
        print("Auto-set EVAL_DIR ->", EVAL_DIR)
    elif len(candidates) > 1:
        # 优先包含当前 patient_id 的路径
        pid = str(LOG_DIR.parts[-1])
        filtered = [c for c in candidates if pid in str(c)]
        if len(filtered) == 1:
            EVAL_DIR = filtered[0].parent
            print("Auto-set EVAL_DIR ->", EVAL_DIR)

image_gt_path = EVAL_DIR / "image_gt.npy"
print("image_gt_path:", image_gt_path)
assert image_gt_path.exists(), "image_gt.npy not found, please update LOG_DIR/EVAL_DIR"

image_gt.npy candidates:
 - /home/jym/naf_cbct/logs/ct_2views_sds0.05_startFrom0_interval10_noLrSchedule/volume-covid19-A-0377_ct/eval/image_gt.npy
 - /home/jym/naf_cbct/logs/ct_2views_sds0.01_startFrom0_interval10_noLrSchedule/volume-covid19-A-0377_ct/eval/image_gt.npy
 - /home/jym/naf_cbct/logs/20260403_CT_1views_centerPad_noLrSchedule_fixVESDEsag_annealing_sds0.05_startFrom0_interval10_nopaas/1.3.6.1.4.1.9328.50.4.0767/eval/image_gt.npy
 - /home/jym/naf_cbct/logs/20260313_CT_2views_sag_centerPad_noLrSchedule_fixVESDEsag_noannealing_sds0.02_startFrom0_interval10/volume-covid19-A-0377_ct/eval/image_gt.npy
 - /home/jym/naf_cbct/logs/nosds_baseline/volume-covid19-A-0377_ct/eval/image_gt.npy
 - /home/jym/naf_cbct/logs/20260313_CT_2views_sag_centerPad_noLrSchedule_fixVESDEsag_noannealing_sds0.05_startFrom0_interval10/volume-covid19-A-0377_ct/eval/image_gt.npy
 - /home/jym/naf_cbct/logs/ct_2views_sds0.1_startFrom0_interval10_noLrSchedule/volume-covid19-A-0377_ct/eval/image_gt.npy
 - /home/

AssertionError: image_gt.npy not found, please update LOG_DIR/EVAL_DIR

## 2. 读取 A(image_gt) 与 projs 数据

In [ ]:
import pickle

# 读取 GT 体积（用于比较）
image_gt = np.load(EVAL_DIR / "image_gt.npy")

# 读取 projs（原始投影）
with open(PICKLE_PATH, "rb") as f:
    data = pickle.load(f)

projs = np.array(data["val"]["projections"], dtype=np.float32)
# 与训练一致：转置 [N,H,W] -> [N,W,H]
projs = np.transpose(projs, (0, 2, 1))

print("image_gt:", image_gt.shape, image_gt.dtype, image_gt.min(), image_gt.max())
print("projs:", projs.shape, projs.dtype, projs.min(), projs.max())

FileNotFoundError: [Errno 2] No such file or directory: '/home/jym/naf_cbct/logs/20260403_CT_1views_centerPad_noLrSchedule_fixVESDEsag_annealing_sds0.005_startFrom0_interval10_nopaas_fidelity2/1.3.6.1.4.1.9328.50.4.0767/eval/image_gt.npy'

## 3. 数据预处理与对齐

In [ ]:
# 选一个角度索引
proj_gt = projs[EVAL_INDEX]

# 强制在 [0,1] 内对齐（如果需要）
image_gt_clipped = np.clip(image_gt, 0.0, 1.0)

print("proj_gt stats:", proj_gt.min(), proj_gt.max(), proj_gt.mean(), proj_gt.std())
print("image_gt stats:", image_gt_clipped.min(), image_gt_clipped.max(), image_gt_clipped.mean(), image_gt_clipped.std())

## 4. 计算差异图与误差指标

In [ ]:
from src.loss.vesde_loss import VESDEGuidance

# 用与训练一致的几何/投影器，构造 A(image_gt)
config_path = "src/sde_configs/ve/IXI_256_ncsnpp_continuous.py"
ckpt_path = "/home/public/CTSpine1K/data/diffusion/checkpoint_ax.pth"

# 初始化 guidance（仅用内部投影器）
vesde = VESDEGuidance(config_path, ckpt_path, device="cuda")

# rays 构造：与训练一致使用 TIGREDataset 取 eval rays
from src.dataset import TIGREDataset

# 使用验证集 rays
dset = TIGREDataset(str(PICKLE_PATH), n_rays=4096, type="val", device="cuda")

rays = dset.rays[EVAL_INDEX].reshape(-1, 8)
bound_box = torch.tensor([dset.geo.sVoxel[0]/2, dset.geo.sVoxel[1]/2, dset.geo.sVoxel[2]/2], device="cuda")

# image_gt -> tensor volume [1,1,D,H,W]
vol = torch.tensor(image_gt_clipped, device="cuda").unsqueeze(0).unsqueeze(0)

# forward project
with torch.no_grad():
    proj_pred = vesde._project_volume_with_rays(vol, rays, bound_box, n_samples=512, rays_chunk=4096)

proj_pred = proj_pred.reshape(proj_gt.shape).cpu().numpy()

# metrics
mse = np.mean((proj_pred - proj_gt)**2)
psnr = 20 * np.log10(1.0 / np.sqrt(mse + 1e-8))
print("MSE:", mse, "PSNR:", psnr)


## 5. 可视化对比与差异热力图

In [ ]:
diff = proj_pred - proj_gt
abs_diff = np.abs(diff)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(proj_gt, cmap="gray")
axes[0].set_title("projs (GT)")
axes[1].imshow(proj_pred, cmap="gray")
axes[1].set_title("A(image_gt)")
axes[2].imshow(abs_diff, cmap="magma")
axes[2].set_title("|diff|")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 6. 保存结果图像

In [ ]:
out_path = EVAL_DIR / "diag_proj_diff.png"
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(proj_gt, cmap="gray")
axes[0].set_title("projs (GT)")
axes[1].imshow(proj_pred, cmap="gray")
axes[1].set_title("A(image_gt)")
axes[2].imshow(abs_diff, cmap="magma")
axes[2].set_title("|diff|")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.savefig(out_path, dpi=200)
print("saved to", out_path)